# 03 · JOINs: INNER JOIN and LEFT JOIN

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

**Tables:** `claims`, `patients`, `providers`, `procedures`.

> 🇪🇸 **Para Josué:** hoy toca **unir tablas**. `claims` solo tiene códigos (`patient_id`, `provider_id`, `cpt_code`); los nombres y descripciones están en las otras 3 tablas. Un JOIN es el `BUSCARV` de Excel: trae datos de otra tabla usando un código en común.
> Hay **4 tablas** (las ves abajo). La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


## The 4 tables

> 🇪🇸 `claims` y `patients` son largas: muestro solo las **primeras 10** filas con `LIMIT 10`.
> Fíjate en los **códigos en común**: `claims.patient_id` ↔ `patients.patient_id`, `claims.provider_id` ↔ `providers.provider_id` y `claims.cpt_code` ↔ `procedures.cpt_code`.

### `claims` (1,000 rows)

In [2]:
duckdb.sql("""
    SELECT * FROM claims LIMIT 10
""").df()

,claim_id,patient_id,provider_id,date_of_service,cpt_code,billed_amount,paid_amount,claim_status,denial_reason
0,CL00001,PA0106,PR011,2026-01-01,71046,337,239,Paid,NaN
1,CL00002,PA0048,PR003,2026-01-01,93306,1015,850,Paid,NaN
2,CL00003,PA0044,PR004,2026-01-01,97140,163,0,Denied,Service not covered
3,CL00004,PA0141,PR010,2026-01-01,99203,204,119,Paid,NaN
4,CL00005,PA0132,PR009,2026-01-01,97110,180,109,Paid,NaN
5,CL00006,PA0108,PR008,2026-01-01,93306,969,580,Paid,NaN
6,CL00007,PA0040,PR005,2026-01-01,85025,59,35,Paid,NaN
7,CL00008,PA0102,PR005,2026-01-02,99214,212,117,Paid,NaN
8,CL00009,PA0177,PR005,2026-01-02,80053,92,66,Paid,NaN
9,CL00010,PA0091,PR009,2026-01-02,97140,156,127,Paid,NaN


### `patients` (200 rows)

In [3]:
duckdb.sql("""
    SELECT * FROM patients LIMIT 10
""").df()

,patient_id,patient_name,date_of_birth,insurance_plan,state
0,PA0001,Laura Lee,1969-01-19,Commercial,TX
1,PA0002,Michael Lopez,1946-07-01,Commercial,AZ
2,PA0003,Susan Johnson,1945-03-15,Commercial,TX
3,PA0004,Jessica Williams,1978-11-26,Medicaid,AZ
4,PA0005,Linda Davis,1958-01-31,Medicare,AZ
5,PA0006,Laura Johnson,1945-07-20,Medicaid,FL
6,PA0007,David Martinez,1944-03-06,Medicaid,NY
7,PA0008,Maria Moore,1952-12-09,Medicaid,FL
8,PA0009,Richard Lopez,1949-03-30,Medicaid,TX
9,PA0010,Patricia Martin,1989-02-20,Commercial,FL


### `providers` (13 rows)

In [4]:
duckdb.sql("""
    SELECT * FROM providers
""").df()

,provider_id,provider_name,specialty,state
0,PR001,Sunrise Family Clinic,Family Medicine,TX
1,PR002,Lakeside Imaging,Radiology,FL
2,PR003,Northside Cardiology,Cardiology,TX
3,PR004,Bayview Physical Therapy,Physical Therapy,CA
4,PR005,Pine Valley Medical Group,Internal Medicine,AZ
5,PR006,Riverside Family Practice,Family Medicine,FL
6,PR007,Desert Sun Radiology,Radiology,AZ
7,PR008,Golden Coast Heart Center,Cardiology,CA
8,PR009,Harbor Rehab & Therapy,Physical Therapy,FL
9,PR010,Maple Street Clinic,Family Medicine,NY


### `procedures` (13 rows)

In [5]:
duckdb.sql("""
    SELECT * FROM procedures
""").df()

,cpt_code,description,service_type,standard_charge
0,99213,"Office visit, established patient, low complexity",Office Visit,150
1,99214,"Office visit, established patient, moderate co...",Office Visit,220
2,99215,"Office visit, established patient, high comple...",Office Visit,300
3,99203,"Office visit, new patient, low complexity",Office Visit,200
4,71046,"Chest X-ray, 2 views",Imaging,320
5,72148,MRI lumbar spine without contrast,Imaging,1200
6,74177,CT abdomen and pelvis with contrast,Imaging,1500
7,93000,"Electrocardiogram (ECG), complete",Cardiology,210
8,93306,"Echocardiogram, complete",Cardiology,950
9,97110,"Therapeutic exercises, 15 minutes",Physical Therapy,180


---
## Example · INNER JOIN

Each claim with the **name** of its patient.

> 🇪🇸 **Te recuerdo:** `INNER JOIN` une dos tablas por un código en común (`ON`) y deja **solo las filas que tienen pareja** en las dos tablas.
> Cuando una columna existe en las dos tablas (aquí `patient_id`), hay que decir de cuál viene: `claims.patient_id`.

In [6]:
duckdb.sql("""
    SELECT claims.claim_id, claims.patient_id, patients.patient_name, claims.billed_amount
    FROM claims
    INNER JOIN patients ON claims.patient_id = patients.patient_id
    ORDER BY claims.claim_id
""").df()

,claim_id,patient_id,patient_name,billed_amount
0,CL00001,PA0106,Jessica Thompson,337
1,CL00002,PA0048,Joseph Miller,1015
2,CL00003,PA0044,Jennifer Thompson,163
3,CL00004,PA0141,James Lewis,204
4,CL00005,PA0132,Anthony Smith,180
...,...,...,...,...
995,CL00996,PA0030,Sofia Martin,188
996,CL00997,PA0122,James Anderson,308
997,CL00998,PA0016,Elizabeth Robinson,294
998,CL00999,PA0190,Maria Clark,64


## Exercise 1 · Claims with provider names

Show every claim with the **name** and **specialty** of its provider. Sort by `claim_id`.

Columns: `claim_id`, `date_of_service`, `provider_name`, `specialty`, `billed_amount`.

✔ **996 rows** (not 1,000!). The first one is **CL00001** · *Empire Diagnostic Imaging* · *Radiology* · $337.

> 🇪🇸 Como el ejemplo, pero con la tabla `providers` (código en común: `provider_id`).
> Cuando lo tengas, piensa: `claims` tiene 1,000 filas, ¿por qué salen 996? (Te acuerdas de PR099 y PR104 del tema 2.)

In [35]:
duckdb.sql("""
    SELECT
    claims.claim_id, claims.date_of_service, providers.provider_name, providers.specialty, claims.billed_amount
    FROM claims
    INNER JOIN providers
    ON claims.provider_id = providers.provider_id
    ORDER BY claims.claim_id
    
""").df()

,claim_id,date_of_service,provider_name,specialty,billed_amount
0,CL00001,2026-01-01,Empire Diagnostic Imaging,Radiology,337
1,CL00002,2026-01-01,Northside Cardiology,Cardiology,1015
2,CL00003,2026-01-01,Bayview Physical Therapy,Physical Therapy,163
3,CL00004,2026-01-01,Maple Street Clinic,Family Medicine,204
4,CL00005,2026-01-01,Harbor Rehab & Therapy,Physical Therapy,180
...,...,...,...,...,...
991,CL00996,2026-06-29,Bayview Physical Therapy,Physical Therapy,188
992,CL00997,2026-06-29,Lakeside Imaging,Radiology,308
993,CL00998,2026-06-30,Northside Cardiology,Cardiology,294
994,CL00999,2026-06-30,Maple Street Clinic,Family Medicine,64


## Exercise 2 · Billing by service type

For each `service_type`, show the number of claims and the total billed, from the highest to the lowest total billed.

Columns: `service_type`, `total_claims`, `total_billed`.

✔ **5 rows.** The first one is **Imaging** (247 claims, $268,741).

> 🇪🇸 `service_type` no está en `claims`: está en `procedures`. Primero el `JOIN` (código en común: `cpt_code`), después `GROUP BY service_type`, como en el tema 2.
> El orden: `SELECT` → `FROM` → `JOIN ... ON` → `GROUP BY` → `ORDER BY`.

In [21]:
duckdb.sql("""
    SELECT 
    procedures.service_type,
    COUNT(*) AS total_claims,
    SUM(billed_amount) AS total_billed
    FROM claims
    INNER JOIN procedures
    ON claims.cpt_code = procedures.cpt_code
    GROUP BY service_type
    ORDER BY total_billed DESC
""").df()

,service_type,total_claims,total_billed
0,Imaging,247,268741.0
1,Office Visit,268,59957.0
2,Cardiology,80,45452.0
3,Physical Therapy,198,34496.0
4,Lab,207,16439.0


---
## New: table aliases

An **alias** is a short nickname for a table, so you don't have to write the full name every time.

> 🇪🇸 **Lo nuevo:** `FROM claims AS c` = "a la tabla `claims` la voy a llamar `c`". Después escribes `c.claim_id` en vez de `claims.claim_id`.
> Es como ponerle un apodo a alguien con nombre largo. En las empresas casi todos lo usan, sobre todo con 3 o 4 JOINs.
> **Es opcional:** si te confunde, sigue escribiendo el nombre completo; el resultado es el mismo.

**Example:** the same query as before, with aliases `c` (claims) and `p` (patients).

In [ ]:
duckdb.sql("""
    SELECT c.claim_id, c.patient_id, p.patient_name, c.billed_amount
    FROM claims AS c
    INNER JOIN patients AS p 
    ON c.patient_id = p.patient_id
    ORDER BY c.claim_id
""").df()

,claim_id,patient_id,patient_name,billed_amount
0,CL00001,PA0106,Jessica Thompson,337
1,CL00002,PA0048,Joseph Miller,1015
2,CL00003,PA0044,Jennifer Thompson,163
3,CL00004,PA0141,James Lewis,204
4,CL00005,PA0132,Anthony Smith,180
...,...,...,...,...
995,CL00996,PA0030,Sofia Martin,188
996,CL00997,PA0122,James Anderson,308
997,CL00998,PA0016,Elizabeth Robinson,294
998,CL00999,PA0190,Maria Clark,64


## Exercise 3 · Medicaid denials

Show the **denied** claims of **Medicaid** patients, oldest first.

Columns: `claim_id`, `date_of_service`, `patient_name`, `billed_amount`, `denial_reason`.

✔ **56 rows.** The first one is **CL00017** (2026-01-03, *Joseph Williams*, $191, *Missing prior authorization*).

> 🇪🇸 `JOIN` de `claims` con `patients`, y dos condiciones en el `WHERE`: una de cada tabla (`claim_status` está en `claims`; `insurance_plan`, en `patients`).
> Puedes usar alias o nombres completos, como prefieras.

In [23]:
duckdb.sql("""
    SELECT c.claim_id, c.date_of_service, pa.patient_name, c.billed_amount, c.denial_reason
    FROM claims as c
    INNER JOIN patients as pa
    ON c.patient_id = pa.patient_id
    WHERE c.claim_status = 'Denied' AND pa.insurance_plan = 'Medicaid'  
    ORDER BY c.date_of_service
""").df()

,claim_id,date_of_service,patient_name,billed_amount,denial_reason
0,CL00017,2026-01-03,Joseph Williams,191,Missing prior authorization
1,CL00097,2026-01-18,Jessica Williams,1616,Missing prior authorization
2,CL00128,2026-01-23,Jessica Williams,1266,Missing prior authorization
3,CL00132,2026-01-24,Laura Taylor,91,Patient not eligible
4,CL00138,2026-01-24,Anthony Taylor,173,Missing prior authorization
5,CL00145,2026-01-25,Laura Gonzalez,154,Service not covered
6,CL00184,2026-01-30,William Lopez,1453,Patient not eligible
7,CL00187,2026-01-31,Emily Harris,65,Duplicate claim
8,CL00193,2026-02-01,Michelle Lewis,173,Missing prior authorization
9,CL00203,2026-02-03,Mary Lee,1565,Service not covered


---
## Example · LEFT JOIN

> 🇪🇸 **Te recuerdo:** `LEFT JOIN` deja **todas** las filas de la tabla de la izquierda (la del `FROM`), tengan o no pareja. Las que no tienen pareja quedan con la columna de la derecha **vacía** (`None` o `NaN`).
> Con `WHERE ... IS NULL` te quedas justo con las que **no** tienen pareja. Es la forma de encontrar lo que "falta".

All patients and their claims; the patients **without** claims would show `claim_id` empty. (Here every patient has claims, so the result is empty.)

In [ ]:
duckdb.sql("""
    SELECT patients.patient_id, patients.patient_name
    FROM patients
    LEFT JOIN claims ON patients.patient_id = claims.patient_id
    WHERE claims.claim_id IS NULL
""").df()

,patient_id,patient_name,claim_id


> 🇪🇸 0 filas = no hay pacientes sin claims. Un resultado vacío también es una respuesta.

## Exercise 4 · Providers without claims

Find the providers that have **no claims** at all.

Columns: `provider_id`, `provider_name`.

✔ **1 row:** **PR013** · *Westfield Physical Therapy*.

> 🇪🇸 Como el ejemplo, pero con `providers` a la izquierda (en el `FROM`) y `claims` a la derecha.

In [ ]:
duckdb.sql("""
    SELECT p.provider_id, p.provider_name
    FROM providers as p
    LEFT JOIN claims as c
    ON p.provider_id = c.provider_id
    WHERE c.claim_id IS NULL
""").df()

,provider_id,provider_name,claim_id
0,PR013,Westfield Physical Therapy,None


## Exercise 5 · Orphan claims

Find the claims whose `provider_id` **does not exist** in the `providers` table. Sort by `claim_id`.

Columns: `claim_id`, `provider_id`, `date_of_service`, `billed_amount`.

✔ **4 rows:** CL00140 (PR099), CL00250 (PR104), CL00419 (PR099), CL00709 (PR104).

> 🇪🇸 Al revés que el ejercicio 4: ahora `claims` va a la izquierda y `providers` a la derecha.
> **Ojo:** el `provider_id` que muestras tiene que salir de `claims` (la izquierda), porque el de `providers` viene vacío cuando no hay pareja.
> En billing, estos son claims que hay que corregir antes de reenviar: el código de proveedor está mal.

In [31]:
duckdb.sql("""
    SELECT c.claim_id, c.provider_id, c.date_of_service, c.billed_amount
    FROM claims as c
    LEFT JOIN providers as p
    ON c.provider_id = p.provider_id
    WHERE p.provider_id IS NULL
    ORDER BY c.claim_id
""").df()

,claim_id,provider_id,date_of_service,billed_amount
0,CL00140,PR099,2026-01-25,334
1,CL00250,PR104,2026-02-12,1581
2,CL00419,PR099,2026-03-14,186
3,CL00709,PR104,2026-05-05,1204


---
## Challenge ⭐⭐⭐ · The most expensive denials, with all the names

Show the **5 most expensive denied claims**, with the patient name, the provider name and the procedure description.

Columns: `claim_id`, `date_of_service`, `patient_name`, `provider_name`, `description`, `billed_amount`, `denial_reason`.

✔ **5 rows.** The first one is **CL00210** · *Jessica Williams* · *Desert Sun Radiology* · *CT abdomen and pelvis with contrast* · $1,646 · *Missing prior authorization*.

> 🇪🇸 **Tres JOINs** seguidos, uno debajo del otro: `claims` con `patients`, con `providers` y con `procedures`. Cada uno con su `ON`.
> Después `WHERE` (solo Denied), `ORDER BY` (de mayor a menor) y `LIMIT 5`.
> Aquí los alias ayudan mucho. Una idea: `c` = claims, `pa` = patients, `pr` = providers, `pc` = procedures.

In [34]:
duckdb.sql("""
    SELECT c.claim_id, c.date_of_service, pa.patient_name, pr.provider_name, pc.description, c.billed_amount, c.denial_reason
    FROM claims as c
    LEFT JOIN patients as pa
    ON c.patient_id = pa.patient_id
    LEFT JOIN procedures as pc
    ON c.cpt_code = pc.cpt_code
    LEFT JOIN providers as pr
    ON c.provider_id = pr.provider_id
    WHERE c.claim_status = 'Denied' 
    ORDER BY c.billed_amount DESC
    LIMIT 5

""").df()

,claim_id,date_of_service,patient_name,provider_name,description,billed_amount,denial_reason
0,CL00210,2026-02-06,Jessica Williams,Desert Sun Radiology,CT abdomen and pelvis with contrast,1646,Missing prior authorization
1,CL00027,2026-01-05,Joseph Miller,Empire Diagnostic Imaging,CT abdomen and pelvis with contrast,1632,Missing prior authorization
2,CL00192,2026-02-01,David Wilson,Lakeside Imaging,CT abdomen and pelvis with contrast,1630,Missing prior authorization
3,CL00579,2026-04-10,John Anderson,Lakeside Imaging,CT abdomen and pelvis with contrast,1622,Service not covered
4,CL00342,2026-02-28,Ana Moore,Desert Sun Radiology,CT abdomen and pelvis with contrast,1620,Missing prior authorization


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. En el ejercicio 1 salieron 996 filas y no 1,000. ¿Qué pasó con las otras 4? ¿Qué JOIN usarías para que salgan las 1,000?
> 2. ¿Qué es un alias y para qué sirve?